# Activity 6: pandas parallel drills

Run each cell top to bottom with the repository-root interpreter. This notebook is the pandas half of Activity 6. The SQL half and full instructions are in `Activity_6_Retail_SQL_and_Pandas_Parallel_Circuit.md`.

Week 4 Day 4 Activity 6: pandas parallel to the retail SQL drills.

Run this file one # %% section at a time in VS Code with the repository-root
interpreter. Each drill is the pandas twin of a drill in
w4d4_retail_cte_and_windows.sql. The message of this activity: the thinking is
the same in SQL and pandas. Only the syntax changes.

Complete the TODOs without editing the provided connection and extraction code.

In [ ]:
# Imports and Snowflake connection, run as written
from configparser import ConfigParser
from pathlib import Path

import pandas as pd
import snowflake.connector


CONFIG_PATH = Path("snow.cfg")
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Complete Activity 0 and keep snow.cfg beside this script."
    )

config = ConfigParser()
config.read(CONFIG_PATH)
if "DEV" not in config:
    raise KeyError("snow.cfg must contain a [DEV] section.")

params = dict(config["DEV"])
if any("<" in value or ">" in value for value in params.values()):
    raise ValueError("Replace every placeholder in snow.cfg before connecting.")

# snow.cfg holds your password and is ignored by the repository-root .gitignore.
# Keep the password in snow.cfg only. Never paste it into this script.
conn = snowflake.connector.connect(**params)


def fetch_dataframe(query: str) -> pd.DataFrame:
    """Run a SELECT query and return lowercase DataFrame columns."""
    with conn.cursor() as cursor:
        cursor.execute(query)
        frame = cursor.fetch_pandas_all()
    frame.columns = frame.columns.str.lower()
    return frame

In [ ]:
# Load the shared practice tables, run as written
stores = fetch_dataframe(
    "SELECT store_id, store_name, region FROM W4D4_STORES ORDER BY store_id"
)
sales = fetch_dataframe(
    """
    SELECT sale_id, store_name, product, sale_date, sales
    FROM W4D4_SALES
    ORDER BY store_name, sale_date, product
    """
)
sales["sales"] = pd.to_numeric(sales["sales"]).astype("float64")
sales["sale_date"] = pd.to_datetime(sales["sale_date"])

assert stores.shape == (5, 3)
assert sales.shape == (20, 5)
print(stores)
print(sales)

In [ ]:
# PA2: store_totals  (SQL twin: Drill A2, GROUP BY store)
# TODO: merge sales with the region column, group to one row per store,
# create transaction_count and total_sales.
store_totals = None

# Uncomment after completing the TODO.
# assert len(store_totals) == 5
# print(store_totals.sort_values("store_name"))

In [ ]:
# PA3: eligible_stores  (SQL twin: Drill A3, WHERE transaction_count >= 4)
# TODO: keep stores with transaction_count >= 4.
eligible_stores = None

# Uncomment after completing the TODO.
# assert sorted(eligible_stores["store_name"]) == ["A", "B", "C", "D"]

In [ ]:
# PA4: region_benchmarks  (SQL twin: Drill A4, AVG of store totals by region)
# TODO: group eligible_stores by region, average total_sales into region_avg_sales.
region_benchmarks = None

# Uncomment after completing the TODO.
# assert len(region_benchmarks) == 2

In [ ]:
# PA5: above_benchmark  (SQL twin: Drill A5, join two grains and compare)
# TODO: merge eligible_stores with region_benchmarks on region.
# TODO: keep total_sales > region_avg_sales, add amount_above_average, sort desc.
above_benchmark = None

# Uncomment after completing the TODO.
# assert above_benchmark["store_name"].tolist() == ["D", "B"]

In [ ]:
# PB1: product_daily_sales  (SQL twin: Drill B1, change the level first)
# TODO: group sales by product and sale_date, sum into daily_sales.
# TODO: sort by product then sale_date so the window drills below are correct.
product_daily_sales = None

# Uncomment after completing the TODO.
# assert len(product_daily_sales) == 10

In [ ]:
# PB2: moving_avg_3d  (SQL twin: Drill B2, rolling window = SQL window frame)
# TODO: within each product, add a 3-day rolling mean of daily_sales.
# Hint: groupby("product")["daily_sales"].transform(
#           lambda x: x.rolling(3, min_periods=1).mean())
# Round to 2 decimals.

# Uncomment after completing the TODO.
# print(product_daily_sales)

In [ ]:
# PB3: prev_day_sales  (SQL twin: Drill B3, shift = SQL LAG)
# TODO: within each product, add prev_day_sales with a one-row shift.

In [ ]:
# PB4: rank_in_store  (SQL twin: Drill B4, DENSE_RANK within store)
# TODO: rank each sale within its store by sales, highest = 1, dense method.
# Hint: groupby("store_name")["sales"].rank(method="dense", ascending=False)

In [ ]:
# PB5: feature versus product average  (SQL twin: Drill B5)
# TODO: add product_avg_sales (mean sales per product, broadcast with transform),
# sales_diff_from_avg (sales minus that average, round 2), and high_sales (sales > 100).

In [ ]:
# PB6: sales_bin  (SQL twin: Drill B6, CASE = a Python function)
# TODO: classify each sale as Low (< 100), Medium (<= 200), or High (> 200),
# then count rows per bin.
bin_counts = None

# Uncomment after completing the TODO.
# assert bin_counts.loc["Low"] == 8
# assert bin_counts.loc["Medium"] == 11
# assert bin_counts.loc["High"] == 1

In [ ]:
# PB7: store_summary  (SQL twin: Drill B7, groupby agg = GROUP BY)
# TODO: one row per store with total_sales, average_sale, highest_sale, lowest_sale.
store_summary = None

# Uncomment after completing the TODO.
# print(store_summary)

In [ ]:
# Parity check: pandas and SQL return the same above-benchmark stores
sql_answer = fetch_dataframe(
    """
    WITH store_totals AS (
      SELECT st.store_name, st.region,
             COUNT(s.sale_id) AS transaction_count,
             SUM(s.sales) AS total_sales
      FROM W4D4_STORES AS st
      JOIN W4D4_SALES AS s ON st.store_name = s.store_name
      GROUP BY st.store_name, st.region
    ),
    eligible_stores AS (
      SELECT * FROM store_totals WHERE transaction_count >= 4
    ),
    region_benchmarks AS (
      SELECT region, AVG(total_sales) AS region_avg_sales
      FROM eligible_stores GROUP BY region
    )
    SELECT e.store_name
    FROM eligible_stores AS e
    JOIN region_benchmarks AS b ON e.region = b.region
    WHERE e.total_sales > b.region_avg_sales
    ORDER BY e.total_sales - b.region_avg_sales DESC
    """
)

# TODO: build pandas_stores and sql_stores as lists in displayed row order.
pandas_stores = None
sql_stores = None

# Uncomment after completing the TODO.
# assert pandas_stores == sql_stores == ["D", "B"]
# print("Parity check passed:", pandas_stores)

In [ ]:
# Close the connection when finished
conn.close()